# Kiểm tra bộ dữ liệu FMB (audit) – bản gộp

Notebook kiểm tra bộ **FMB** (SegMiF, ICCV 2023), là bộ dữ liệu chính mà SAGE dùng để train và đánh giá phân đoạn.

**Thông tin công bố trong bài SegMiF:** 1.500 cặp ảnh hồng ngoại – ảnh thường đã căn chỉnh, độ phân giải 800×600, nhãn phân đoạn mức pixel cho 14 lớp (bài ghi chỗ 14 chỗ 15, vì có tính hay không tính lớp nền). Ảnh hồng ngoại được chiếu về toạ độ ảnh thường bằng homography rồi cắt.

**Notebook trả lời các câu:**
1. Cấu trúc thư mục ra sao?
2. Mỗi split có bao nhiêu ảnh; IR, VIS, Label có ghép cặp đủ không?
3. Kích thước, số kênh, ảnh hỏng; cặp IR–VIS có cùng kích thước không?
4. Label có giá trị nào, mỗi lớp chiếm bao nhiêu pixel; thư mục `color` là gì?
5. Train và test có rò dữ liệu không?
6. IR và VIS có thật sự căn khớp không?
7. 5 bộ ảnh minh hoạ IR / VIS / Label.
8. Tóm tắt kết quả.

Bản này gộp mục 9 của bản v2 vào các mục 4–7, bỏ phần trùng. **Chạy một lượt từ đầu đến cuối.** Kết quả lưu ở `OUT_DIR` (thư mục mới `fmb_audit_final`, tách khỏi kết quả cũ). Tên file bắt đầu bằng số mục.

## 0. Cấu hình
Sửa `FMB_ROOT` trỏ tới thư mục chứa FMB sau khi giải nén. Nếu để `None`, notebook tự tìm trong `/kaggle/input`, `/content` và thư mục hiện tại.

In [ ]:
import os, re, glob, hashlib, json, random
from collections import Counter, defaultdict
import numpy as np, pandas as pd, cv2
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from tqdm.auto import tqdm

FMB_ROOT = None                 # để None trên Colab (ô sau tự lấy từ Drive). Ví dụ: '/content/FMB'
OUT_DIR  = './fmb_audit_final'  # nơi lưu bảng và hình (trên Colab tự đổi sang Drive)
N_SAMPLES = 5                   # số bộ ảnh minh hoạ
N_ALIGN = 300                   # số cặp dùng để kiểm căn chỉnh IR–VIS
HAM_THR = 5                     # ngưỡng Hamming dHash (trên 64 bit) để coi là gần trùng
SEED = 0
random.seed(SEED); np.random.seed(SEED)

# Tên lớp FMB. Thứ tự đã kiểm ở lần chạy 3/11: khớp ngữ nghĩa trên hình minh hoạ và khớp thứ tự lớp trong train.py của SAGE.
CLASS_NAMES = ['unlabeled', 'road', 'sidewalk', 'building', 'traffic_light', 'traffic_sign',
               'vegetation', 'sky', 'person', 'car', 'truck', 'bus', 'motorcycle', 'bicycle', 'pole']
IMG_EXT = ('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff')

# Ảnh test rò dữ liệu, ĐÃ CHỐT BẰNG MẮT ngày 3/11 (xem mục 5). dHash chỉ dùng để tìm ứng viên.
# Mỗi cụm: (ảnh test, ảnh train cùng cảnh). Hình ở mục 5 ghép mỗi ảnh test với ảnh train gần nhất TRONG cụm.
LEAK_CONFIRMED = {
    'Cụm A – đường đêm, toà nhà + vách đá': (['00587', '00588', '00589', '00590'], [f'{i:05d}' for i in range(489, 501)]),
    'Cụm B – phố chung cư có sương':         (['00631', '00632', '00633'], ['00586']),
}

### Lấy dữ liệu FMB (Colab)
**Trước khi chạy:** mở link FMB https://drive.google.com/drive/folders/1T_jVi80tjgyHTQDpn-TjfySyW4CK1LlF
- Nếu `train`, `test` là file **.zip**: chuột phải từng file → **Tạo bản sao**.
- Nếu là **thư mục**: chuột phải thư mục FMB → **Sắp xếp → Thêm lối tắt → Drive của tôi**.

Ô dưới tự mount Drive, tìm `train`/`test` (ưu tiên đường dẫn có chữ "fmb") rồi giải nén/chép về `/content/FMB`, theo dạng `/content/FMB/train/Visible/...`. Chạy lại lần 2 sẽ bỏ qua vì đã có dữ liệu. Trên Kaggle, ô này tự bỏ qua; khi đó *Add Input* bộ FMB vào notebook.

In [ ]:
try:
    import google.colab; IN_COLAB = True    # nhận biết Colab bằng thư viện riêng của Colab
except ImportError:
    IN_COLAB = False
if IN_COLAB and FMB_ROOT is None:
    import shutil, zipfile
    from google.colab import drive
    drive.mount('/content/drive')
    DST, DRIVE = '/content/FMB', '/content/drive/MyDrive'
    key = lambda p: ('fmb' not in p.lower(), len(p))   # ưu tiên đường dẫn có "fmb"
    for split in ['train', 'test']:
        out = f'{DST}/{split}'
        if os.path.isdir(out) and os.listdir(out):
            print(f'[{split}] đã có ở {out}, bỏ qua'); continue
        zips = sorted([z for z in glob.glob(f'{DRIVE}/**/*.zip', recursive=True) if split in os.path.basename(z).lower()], key=key)
        dirs = sorted([d for d in glob.glob(f'{DRIVE}/**/{split}', recursive=True) if os.path.isdir(d)], key=key)
        print(f'[{split}] zip:', zips[:3], '| thư mục:', dirs[:3])
        if zips:   print('  → giải nén', zips[0]); zipfile.ZipFile(zips[0]).extractall(out)
        elif dirs: print('  → chép', dirs[0]); shutil.copytree(dirs[0], out, dirs_exist_ok=True)
        else:      print(f'  → KHÔNG TÌM THẤY {split}: kiểm tra lại bước tạo bản sao / lối tắt')
        nested = f'{out}/{split}'    # zip có sẵn thư mục cùng tên: đưa lên một cấp, tránh train/train
        if os.path.isdir(nested):
            for name in os.listdir(nested): shutil.move(f'{nested}/{name}', out)
            os.rmdir(nested)
    FMB_ROOT = DST
    OUT_DIR = '/content/drive/MyDrive/fmb_audit_final'   # lưu kết quả lên Drive để không mất
os.makedirs(OUT_DIR, exist_ok=True)
print('FMB_ROOT =', FMB_ROOT, '| OUT_DIR =', OUT_DIR)

## 1. Cấu trúc thư mục
Tự nhận diện thư mục ảnh thường / hồng ngoại / nhãn theo tên (Visible, Infrared, Label, …) và split theo thư mục cha (train, test). Thư mục `color` **không** được coi là ảnh thường: ở FMB đó là nhãn tô màu (kiểm ở mục 4).

In [ ]:
PAT = {
    'vis':   re.compile(r'^(vis|visible|vi|rgb)$', re.I),   # KHÔNG gồm 'color': ở FMB đó là nhãn tô màu
    'ir':    re.compile(r'^(ir|infrared|inf|thermal|t)$', re.I),
    'label': re.compile(r'^(label|labels|gt|mask|masks|seg|segmentation|labelid)$', re.I),
}

def find_root():
    if FMB_ROOT: return FMB_ROOT
    for base in ['/kaggle/input', '/content', '.', './FMB_download']:
        if not os.path.isdir(base): continue
        for dirpath, dirnames, _ in os.walk(base):
            kinds = {k for d in dirnames for k, p in PAT.items() if p.match(d)}
            if {'vis', 'ir'} <= kinds:
                # gốc = thư mục chứa các split (lùi 1 cấp nếu thư mục này là train/test)
                return os.path.dirname(dirpath) if re.search(r'(train|test|val)', os.path.basename(dirpath), re.I) else dirpath
    raise FileNotFoundError('Không tìm thấy FMB. Hãy đặt FMB_ROOT bằng tay.')

ROOT = find_root()
print('FMB_ROOT =', ROOT)

def list_images(d):
    return sorted(f for f in os.listdir(d) if f.lower().endswith(IMG_EXT))
def stem(f): return os.path.splitext(f)[0]

# Gom các thư mục modality theo split
splits = defaultdict(dict)
for dirpath, dirnames, _ in os.walk(ROOT):
    for d in dirnames:
        for kind, p in PAT.items():
            if p.match(d):
                rel = os.path.relpath(dirpath, ROOT)
                split = 'all' if rel == '.' else rel.replace(os.sep, '/')
                splits[split][kind] = os.path.join(dirpath, d)
splits = {s: v for s, v in splits.items() if 'vis' in v and 'ir' in v}

# In cây thư mục (2 cấp) để ghi vào báo cáo
for dirpath, dirnames, files in os.walk(ROOT):
    depth = os.path.relpath(dirpath, ROOT).count(os.sep) + (0 if dirpath == ROOT else 1)
    if depth > 3: continue
    n_img = sum(f.lower().endswith(IMG_EXT) for f in files)
    print('  ' * depth + (os.path.basename(dirpath) or dirpath) + '/' + (f'  ({n_img} ảnh)' if n_img else ''))
print('\nSplit tìm được:', {s: list(v) for s, v in splits.items()})

## 2. Số ảnh và ghép cặp IR–VIS–Label
Một **cặp đủ** là bộ có cùng tên ở cả Visible, Infrared và Label. Ảnh nào thiếu bạn đồng hành được ghi vào `2_loi_ghep_cap.csv`. File này **luôn được tạo**; nếu không có lỗi thì file chỉ có dòng tiêu đề.

In [ ]:
rows, pair_issues, pairs = [], [], {}
for split, dirs in sorted(splits.items()):
    names = {k: {stem(f): f for f in list_images(d)} for k, d in dirs.items()}
    vis, ir = set(names['vis']), set(names['ir'])
    lab = set(names.get('label', {}))
    common = vis & ir & (lab if 'label' in names else vis)
    for k, s in [('vis', vis), ('ir', ir)] + ([('label', lab)] if 'label' in names else []):
        for n in sorted(s - common):
            pair_issues.append({'split': split, 'modality': k, 'file': names[k][n], 'van_de': 'không có cặp'})
    pairs[split] = [{k: os.path.join(dirs[k], names[k][n]) for k in names} | {'name': n} for n in sorted(common)]
    rows.append({'split': split, 'VIS': len(vis), 'IR': len(ir), 'Label': len(lab) if 'label' in names else None,
                 'Cặp đủ': len(common), 'Lỗi ghép cặp': len(vis | ir | lab) - len(common)})

df_count = pd.DataFrame(rows)
total = df_count[['VIS', 'IR', 'Cặp đủ']].sum()
df_count.loc[len(df_count)] = {'split': 'TỔNG', 'VIS': total['VIS'], 'IR': total['IR'],
                               'Label': df_count['Label'].sum(), 'Cặp đủ': total['Cặp đủ'],
                               'Lỗi ghép cặp': df_count['Lỗi ghép cặp'].sum()}
df_count.to_csv(f'{OUT_DIR}/2_so_anh_theo_split.csv', index=False)
display(df_count)
print('Theo bài công bố: 1.500 cặp. Tổng cặp đủ ở đây:', int(total['Cặp đủ']))

df_pi = pd.DataFrame(pair_issues, columns=['split', 'modality', 'file', 'van_de'])
df_pi.to_csv(f'{OUT_DIR}/2_loi_ghep_cap.csv', index=False)
if len(df_pi): display(df_pi.head(20))
else: print('✓ 0 lỗi ghép cặp (2_loi_ghep_cap.csv chỉ có dòng tiêu đề).')

path_of = {(s, p['name']): p for s, pl in pairs.items() for p in pl}
all_pairs = [(s, p) for s, pl in pairs.items() for p in pl]
split_names = list(pairs)
test_s  = next((s for s in split_names if 'test'  in s.lower()), None)
train_s = next((s for s in split_names if 'train' in s.lower()), None)

## 3. Kích thước, số kênh, ảnh hỏng
Đọc toàn bộ ảnh một lần, ghi lại kích thước và số kênh, kiểm xem IR 3 kênh có phải 3 kênh giống nhau không. Đồng thời tính MD5 (trùng tuyệt đối) và dHash của **cả VIS lẫn IR** (gần trùng) để dùng ở mục 5.

In [ ]:
def dhash(gray, size=8):
    r = cv2.resize(gray, (size + 1, size), interpolation=cv2.INTER_AREA)
    return np.packbits((r[:, 1:] > r[:, :-1]).flatten())
def to_gray(img): return img if img.ndim == 2 else cv2.cvtColor(img[..., :3], cv2.COLOR_BGR2GRAY)

recs, broken = [], []
for split, plist in pairs.items():
    for p in tqdm(plist, desc=split):
        r = {'split': split, 'name': p['name']}
        for k in ('vis', 'ir', 'label'):
            if k not in p: continue
            img = cv2.imread(p[k], cv2.IMREAD_UNCHANGED)
            if img is None:
                broken.append({'split': split, 'file': p[k]}); r[f'{k}_ok'] = False; continue
            r[f'{k}_h'], r[f'{k}_w'] = img.shape[:2]
            r[f'{k}_c'] = 1 if img.ndim == 2 else img.shape[2]
            r[f'{k}_dtype'] = str(img.dtype)
            if k == 'vis':
                g = to_gray(img)
                r['vis_md5'] = hashlib.md5(open(p[k], 'rb').read()).hexdigest()
                r['vis_dhash'] = dhash(g)
                r['vis_mean'] = float(g.mean())   # độ sáng trung bình: phân biệt ngày / đêm
            if k == 'ir':
                g = to_gray(img)
                r['ir_dhash'] = dhash(g)
                r['ir_mean'] = float(g.mean())
                r['ir_3kenh_giong'] = bool(img.ndim == 3 and np.array_equal(img[..., 0], img[..., 1]) and np.array_equal(img[..., 1], img[..., 2]))
        recs.append(r)
df_img = pd.DataFrame(recs)

size_cols = [c for c in df_img.columns if re.match(r'(vis|ir|label)_(h|w|c|dtype)$', c)]
print('Các tổ hợp kích thước / kênh / kiểu dữ liệu:')
display(df_img.groupby(['split'] + size_cols).size().rename('số ảnh').reset_index())

same = (df_img['vis_h'] == df_img['ir_h']) & (df_img['vis_w'] == df_img['ir_w'])
if 'label_h' in df_img:
    same &= (df_img['vis_h'] == df_img['label_h']) & (df_img['vis_w'] == df_img['label_w'])
print(f'Cặp lệch kích thước: {(~same).sum()}  |  Ảnh không đọc được: {len(broken)}')
n_ir3 = int(df_img['ir_3kenh_giong'].sum()) if 'ir_3kenh_giong' in df_img else 0
print(f'IR lưu 3 kênh nhưng 3 kênh giống hệt: {n_ir3}/{len(df_img)} → khi train chỉ cần đọc 1 kênh.')
df_img.drop(columns=['vis_dhash', 'ir_dhash']).to_csv(f'{OUT_DIR}/3_thong_tin_tung_anh.csv', index=False)
if broken: pd.DataFrame(broken).to_csv(f'{OUT_DIR}/3_anh_hong.csv', index=False)

## 4. Nhãn phân đoạn và thư mục `color`
**4a.** Kiểm label là ảnh chỉ số lớp (1 kênh, giá trị 0..14). Đếm, cho mỗi lớp: tỉ lệ pixel và số ảnh có chứa lớp đó. Chỉ ra lớp có ở train mà không có ở test.

**4b.** Thư mục `color`: kiểm xem mỗi id trong Label có ứng đúng 1 màu trong `color` không (trên 40 ảnh mỗi split). Nếu đúng thì `color` là nhãn tô màu theo bảng màu gốc của FMB, và bảng màu này được dùng để vẽ hình ở mục 7.

In [ ]:
def read_label(path):
    lab = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    if lab.ndim == 3:
        if np.array_equal(lab[..., 0], lab[..., 1]) and np.array_equal(lab[..., 1], lab[..., 2]):
            lab = lab[..., 0]          # 3 kênh giống nhau → thực chất là ảnh chỉ số
        else:
            raise ValueError('Label là ảnh MÀU: cần bảng màu → chỉ số trước khi đếm.')
    return lab
def cname(v): return CLASS_NAMES[v] if v < len(CLASS_NAMES) else f'? ({v})'

# ---- 4a. Phân bố lớp ----
pix = defaultdict(Counter); img_cnt = defaultdict(Counter)
for split, plist in pairs.items():
    for p in tqdm(plist, desc=f'label {split}'):
        if 'label' not in p: continue
        v, c = np.unique(read_label(p['label']), return_counts=True)
        for vi, ci in zip(v.tolist(), c.tolist()):
            pix[split][vi] += ci; img_cnt[split][vi] += 1
values = sorted(set().union(*[set(c) for c in pix.values()]))
print('Giá trị pixel xuất hiện trong label:', values)
if max(values) >= len(CLASS_NAMES):
    print(f'⚠ Có giá trị ≥ {len(CLASS_NAMES)} (ví dụ 255 = ignore). Kiểm lại cách mã hoá.')

rows = []
for v in values:
    r = {'id': v, 'lớp': cname(v)}
    for split in pix:
        tot = sum(pix[split].values())
        r[f'{split}_%pixel'] = round(100 * pix[split][v] / tot, 3)
        r[f'{split}_số_ảnh'] = img_cnt[split][v]
    rows.append(r)
df_cls = pd.DataFrame(rows); df_cls.to_csv(f'{OUT_DIR}/4_phan_bo_lop.csv', index=False)
display(df_cls)

fig, ax = plt.subplots(figsize=(11, 4))
w = 0.8 / len(pix)
for i, split in enumerate(pix):
    ax.bar(np.arange(len(values)) + i * w, df_cls[f'{split}_%pixel'].clip(lower=1e-4), w, label=split)
ax.set_xticks(np.arange(len(values)) + w * (len(pix) - 1) / 2)
ax.set_xticklabels(df_cls['lớp'], rotation=45, ha='right'); ax.set_yscale('log')
ax.set_ylabel('% pixel (log)'); ax.set_title('Phân bố lớp FMB theo pixel'); ax.legend()
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/4_phan_bo_lop.png', dpi=150); plt.show()

missing = {}
for a in pix:
    for b in pix:
        if a != b:
            miss = [cname(v) for v in values if img_cnt[a][v] > 0 and img_cnt[b][v] == 0]
            if miss: missing[f'co_o_{a}_khong_co_o_{b}'] = miss; print(f'⚠ Lớp có ở {a} nhưng KHÔNG có ở {b}: {miss}')
rare = df_cls[[c for c in df_cls if c.endswith('%pixel')]].max(axis=1) < 0.5
print('Lớp hiếm (< 0,5% pixel ở mọi split):', df_cls.loc[rare, 'lớp'].tolist())

# ---- 4b. Thư mục color ----
def read_bgr(path):
    im = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    if im is None: return None
    if im.ndim == 2: im = cv2.cvtColor(im, cv2.COLOR_GRAY2BGR)
    return im[..., :3]

color_dirs = {}
for split, dirs in splits.items():
    base = os.path.dirname(dirs['vis'])
    cands = [d for d in os.listdir(base) if d.lower() == 'color' and os.path.isdir(os.path.join(base, d))]
    if cands: color_dirs[split] = os.path.join(base, cands[0])

OFFICIAL_PALETTE = None
if not color_dirs:
    print('Không thấy thư mục color.')
else:
    pair_cnt = Counter()                       # (id, mã màu) -> số pixel
    for split, cdir in color_dirs.items():
        cmap = {stem(f): f for f in list_images(cdir)}
        names = {p['name'] for p in pairs[split]}
        print(f'{split}: {len(cmap)} ảnh trong color | trùng tên với cặp đủ: {len(names & set(cmap))}/{len(names)}')
        for p in random.Random(SEED).sample(pairs[split], min(40, len(pairs[split]))):
            if p['name'] not in cmap or 'label' not in p: continue
            col = read_bgr(os.path.join(cdir, cmap[p['name']])); lab = read_label(p['label'])
            if col is None or col.shape[:2] != lab.shape[:2]:
                print('  bỏ qua (không đọc được / lệch kích thước):', p['name']); continue
            code_ = (col[..., 2].astype(np.int64) << 16) | (col[..., 1].astype(np.int64) << 8) | col[..., 0]
            k, c = np.unique(lab.astype(np.int64) * (1 << 24) + code_, return_counts=True)
            for ki, ci in zip(k.tolist(), c.tolist()): pair_cnt[(ki >> 24, ki & 0xFFFFFF)] += ci

    rows = []
    for v in sorted({i for i, _ in pair_cnt}):
        cs = {cc: n for (i, cc), n in pair_cnt.items() if i == v}
        top, n_top = max(cs.items(), key=lambda t: t[1])
        rows.append({'id': v, 'lớp': cname(v), 'RGB chính': ((top >> 16) & 255, (top >> 8) & 255, top & 255),
                     '% pixel đúng màu chính': round(100 * n_top / sum(cs.values()), 2), 'số màu khác nhau': len(cs)})
    df_col = pd.DataFrame(rows); display(df_col)
    df_col.to_csv(f'{OUT_DIR}/4_bang_mau_color.csv', index=False)
    not_checked = [cname(v) for v in values if v not in set(df_col['id'])]
    if not_checked: print('Id không có trong mẫu nên chưa kiểm được màu:', not_checked)

    if (df_col['% pixel đúng màu chính'] >= 99).all() and df_col['RGB chính'].nunique() == len(df_col):
        print('✓ Mỗi id ứng đúng 1 màu và các màu khác nhau → color = nhãn tô màu theo bảng màu gốc của FMB.')
        OFFICIAL_PALETTE = np.zeros((256, 3), np.uint8)
        for _, r in df_col.iterrows(): OFFICIAL_PALETTE[r['id']] = r['RGB chính']
    else:
        print('⚠ Không ánh xạ 1–1: xem các dòng có % thấp / nhiều màu, và mở vài ảnh color bằng mắt.')

# Bảng màu dùng để vẽ: gốc nếu có, nếu không thì ngẫu nhiên
rng = np.random.default_rng(SEED)
PALETTE = rng.integers(40, 255, (256, 3)).astype(np.uint8); PALETTE[0] = 0
if OFFICIAL_PALETTE is not None:
    PALETTE[values] = OFFICIAL_PALETTE[values]

## 5. Rò dữ liệu giữa train và test
Ảnh test giống ảnh train (ví dụ các khung liền nhau cắt từ cùng một video) làm kết quả trên test bị đánh giá cao hơn thực tế. CS406 coi data leakage là lỗi nặng.

**Cách làm:**
1. **Trùng tuyệt đối:** cùng MD5.
2. **Tìm ứng viên gần trùng:** khoảng cách Hamming của dHash (64 bit) trên **VIS** và trên **IR**, ở nhiều ngưỡng.
3. **Xác nhận bằng mắt:** dHash IR nhiễu ở ảnh đêm (ví dụ test 00631 ↔ train 00586 trông giống hệt cả trên IR nhưng lệch 11 bit), còn ngưỡng 8–10 bit thì quá lỏng (131–312 cặp, phần lớn báo nhầm). Vì vậy danh sách cuối được **chốt bằng mắt** (3/11) và ghi cứng ở `LEAK_CONFIRMED` (ô cấu hình). Hình cuối mục này cho thấy lại các cặp đó để người đọc tự kiểm.

Các cặp báo nhầm đã loại khi xem bằng mắt (cảnh khác hẳn): test 00632/00633 ↔ train 00442; test 00880 ↔ train 00571.

In [ ]:
assert test_s and train_s, 'Cần có cả split train và test để kiểm rò dữ liệu.'
def ham(col, a, b):
    A = np.stack(df_img.loc[df_img.split == a, col].values); B = np.stack(df_img.loc[df_img.split == b, col].values)
    return np.unpackbits(A[:, None, :] ^ B[None, :, :], axis=2).sum(2)
Hv, Hi = ham('vis_dhash', test_s, train_s), ham('ir_dhash', test_s, train_s)
Nt = df_img.loc[df_img.split == test_s, 'name'].values; Nr = df_img.loc[df_img.split == train_s, 'name'].values

# 1. Trùng tuyệt đối
M = {s: df_img.loc[df_img.split == s, 'vis_md5'] for s in (train_s, test_s)}
n_exact = len(set(M[train_s]) & set(M[test_s]))
print(f'Trùng tuyệt đối train ↔ test: {n_exact}')
for s in (train_s, test_s): print(f'Trùng tuyệt đối bên trong {s}: {M[s].duplicated().sum()}')

# 2. Ứng viên theo ngưỡng
rows = []
for t in [3, 5, 8, 10]:
    v_, i_ = Hv <= t, Hi <= t
    rows.append({'ngưỡng Hamming': t, 'cặp (VIS)': int(v_.sum()), 'cặp (IR)': int(i_.sum()), 'cặp (cả hai)': int((v_ & i_).sum()),
                 'ảnh test dính (VIS)': int(v_.any(1).sum()), 'ảnh test dính (cả hai)': int((v_ & i_).any(1).sum())})
df_th = pd.DataFrame(rows); display(df_th); df_th.to_csv(f'{OUT_DIR}/5_ro_theo_nguong.csv', index=False)

cand = pd.DataFrame([{'anh_test': Nt[a], 'anh_train': Nr[b], 'hamming_vis': int(Hv[a, b]), 'hamming_ir': int(Hi[a, b])}
                     for a, b in np.argwhere((Hv <= HAM_THR) | (Hi <= HAM_THR))],
                    columns=['anh_test', 'anh_train', 'hamming_vis', 'hamming_ir'])
cand = cand.sort_values(['anh_test', 'hamming_vis']).reset_index(drop=True)
print(f'Ứng viên (VIS hoặc IR ≤ {HAM_THR} bit): {len(cand)} cặp, {cand.anh_test.nunique()} ảnh test')
display(cand.head(30)); cand.to_csv(f'{OUT_DIR}/5_ung_vien_ro_vis_ir.csv', index=False)

# 3. Danh sách chốt bằng mắt
num = lambda n: int(re.sub(r'\D', '', n) or -1)            # so theo số để không phụ thuộc định dạng tên file
test_by_num = {num(n): n for n in Nt}
train_idx = {num(n): j for j, n in enumerate(Nr)}
leak_test, leak_rows = [], []
for cum, (lst, train_lst) in LEAK_CONFIRMED.items():
    cols = [train_idx[num(t)] for t in train_lst if num(t) in train_idx]
    assert cols, f'Không thấy ảnh train nào của {cum} – kiểm lại tên file'
    for n in lst:
        name = test_by_num.get(num(n))
        if name is None: print(f'⚠ Không thấy ảnh test {n} trong dữ liệu – kiểm lại tên file'); continue
        a = list(Nt).index(name)
        b = cols[int(np.argmin(Hv[a, cols]))]               # ảnh train gần nhất theo VIS, chỉ trong cụm
        leak_test.append(name)
        leak_rows.append({'cum': cum, 'anh_test': name, 'anh_train_gan_nhat': Nr[b],
                          'hamming_vis': int(Hv[a, b]), 'hamming_ir': int(Hi[a, b])})
df_leak = pd.DataFrame(leak_rows); display(df_leak)
df_leak.to_csv(f'{OUT_DIR}/5_anh_test_ro_da_chot.csv', index=False)
open(f'{OUT_DIR}/5_anh_test_ro.txt', 'w').write('\n'.join(leak_test))
print(f'→ Ảnh test rò (chốt bằng mắt): {len(leak_test)}/{len(Nt)} = {100 * len(leak_test) / len(Nt):.1f}%. '
      f'Danh sách ở 5_anh_test_ro.txt; khi có model tính mIoU trên đủ {len(Nt)} ảnh và trên {len(Nt) - len(leak_test)} ảnh còn lại.')

In [ ]:
# Hình xác nhận: mỗi hàng = ảnh test rò | ảnh train gần nhất, cả VIS và IR
if len(df_leak):
    fig, axes = plt.subplots(len(df_leak), 4, figsize=(16, 2.9 * len(df_leak)), squeeze=False)
    for r, row in df_leak.reset_index(drop=True).iterrows():
        pt, pr = path_of[(test_s, row.anh_test)], path_of[(train_s, row.anh_train_gan_nhat)]
        for c, (pp, mod, tag) in enumerate([(pt, 'vis', f'test {row.anh_test}'), (pr, 'vis', f'train {row.anh_train_gan_nhat}'),
                                            (pt, 'ir', f'test {row.anh_test}'), (pr, 'ir', f'train {row.anh_train_gan_nhat}')]):
            axes[r, c].imshow(cv2.cvtColor(read_bgr(pp[mod]), cv2.COLOR_BGR2RGB))
            ham_ = row.hamming_vis if mod == 'vis' else row.hamming_ir
            axes[r, c].set_title(f'{mod.upper()} {tag}  (Hamming {ham_})', fontsize=8); axes[r, c].axis('off')
    plt.tight_layout(); plt.savefig(f'{OUT_DIR}/5_xem_mat_anh_ro.png', dpi=110); plt.show()

## 6. IR và VIS có căn khớp không?
Đo độ dịch giữa bản đồ cạnh (Sobel) của IR và VIS bằng tương quan pha (`cv2.phaseCorrelate`) trên **300 cặp ngẫu nhiên**.
- `dx`, `dy` là độ dịch của IR so với VIS (pixel): `dx > 0` là IR lệch sang phải, `dy > 0` là IR lệch xuống. Nếu lệch **có hệ thống** thì trung vị `dx` hoặc `dy` khác 0 rõ (ví dụ +3 px). Nếu chỉ là nhiễu đo thì trung vị gần 0.
- `do_tin_cay` là đỉnh tương quan pha. Ảnh tối, sương mù, ít cạnh cho độ tin cậy thấp và độ dịch vô nghĩa, nên **chỉ kết luận trên nửa có độ tin cậy cao**.
- Phép đo này không thấy được lệch xoay hay lệch cục bộ. Vì vậy cuối mục vẽ chồng cạnh IR lên 3 cặp lệch nhất trong nhóm tin cậy cao để xem bằng mắt.

In [ ]:
def edge(img):
    g = cv2.GaussianBlur(to_gray(img).astype(np.float32), (5, 5), 0)
    e = cv2.magnitude(cv2.Sobel(g, cv2.CV_32F, 1, 0), cv2.Sobel(g, cv2.CV_32F, 0, 1))
    return (e - e.mean()) / (e.std() + 1e-6)

def overlay_edges(vis, ir):
    g = cv2.GaussianBlur(to_gray(ir), (5, 5), 0)
    m = float(np.median(g)); e = cv2.Canny(g, int(max(0, 0.66 * m)), int(min(255, 1.33 * m) + 20))  # ngưỡng tự động theo độ sáng
    out = cv2.cvtColor(vis, cv2.COLOR_BGR2RGB).copy(); out[e > 0] = (255, 0, 0); return out

rs = random.Random(SEED + 1)
align = []
for s, p in tqdm(rs.sample(all_pairs, min(N_ALIGN, len(all_pairs))), desc='căn chỉnh'):
    v, i = cv2.imread(p['vis']), cv2.imread(p['ir'])
    if v is None or i is None or v.shape[:2] != i.shape[:2]: continue
    win = cv2.createHanningWindow(v.shape[1::-1], cv2.CV_32F)
    (dx, dy), resp = cv2.phaseCorrelate(edge(v) * win, edge(i) * win)
    align.append({'split': s, 'name': p['name'], 'dx': dx, 'dy': dy, 'do_tin_cay': resp})
df_al = pd.DataFrame(align).merge(df_img[['split', 'name', 'vis_mean']], on=['split', 'name'])
df_al['lech_px'] = np.hypot(df_al.dx, df_al.dy)
df_al['tin_cay_cao'] = df_al['do_tin_cay'] >= df_al['do_tin_cay'].median()
df_al.to_csv(f'{OUT_DIR}/6_can_chinh_tung_cap.csv', index=False)

def tk(d):
    return pd.Series({'số cặp': len(d), 'trung vị dx': d.dx.median(), 'trung vị dy': d.dy.median(),
                      'trung vị |lệch|': d.lech_px.median(), 'P90 |lệch|': d.lech_px.quantile(0.9),
                      '% > 3 px': 100 * (d.lech_px > 3).mean(), '% > 10 px': 100 * (d.lech_px > 10).mean(),
                      'độ sáng VIS TB': d.vis_mean.mean()})
bang = pd.DataFrame({'tất cả': tk(df_al), 'tin cậy cao': tk(df_al[df_al.tin_cay_cao]),
                     'tin cậy thấp': tk(df_al[~df_al.tin_cay_cao])}).round(2)
display(bang); bang.to_csv(f'{OUT_DIR}/6_tom_tat_can_chinh.csv')
corr_tc_sang = round(df_al[['do_tin_cay', 'vis_mean']].corr().iloc[0, 1], 2)
print('Tương quan độ tin cậy – độ sáng VIS:', corr_tc_sang, '(dương → ảnh tối đo kém tin cậy)')

fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
sc = ax[0].scatter(df_al.dx.clip(-20, 20), df_al.dy.clip(-20, 20), c=df_al.do_tin_cay, cmap='viridis', s=14)
ax[0].axhline(0, c='k', lw=.5); ax[0].axvline(0, c='k', lw=.5); plt.colorbar(sc, ax=ax[0], label='độ tin cậy')
ax[0].set_xlabel('dx (px, cắt ở ±20)'); ax[0].set_ylabel('dy (px)'); ax[0].set_title('Độ dịch IR so với VIS')
ax[1].hist([df_al.loc[df_al.tin_cay_cao, 'lech_px'].clip(0, 30), df_al.loc[~df_al.tin_cay_cao, 'lech_px'].clip(0, 30)],
           bins=30, label=['tin cậy cao', 'tin cậy thấp'], stacked=True)
ax[1].set_xlabel('|lệch| (px, cắt ở 30)'); ax[1].set_ylabel('số cặp'); ax[1].legend(); ax[1].set_title('Phân bố độ lệch')
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/6_can_chinh.png', dpi=130); plt.show()

worst = df_al[df_al.tin_cay_cao].nlargest(3, 'lech_px')
if len(worst):
    fig, axes = plt.subplots(1, len(worst), figsize=(5 * len(worst), 3.8), squeeze=False)
    for c, (_, r) in enumerate(worst.iterrows()):
        p = path_of[(r.split, r['name'])]
        axes[0, c].imshow(overlay_edges(cv2.imread(p['vis']), cv2.imread(p['ir'])))
        axes[0, c].set_title(f"{r.split}/{r['name']}  dx={r.dx:.1f} dy={r.dy:.1f}  tin cậy {r.do_tin_cay:.2f}", fontsize=8)
        axes[0, c].axis('off')
    plt.tight_layout(); plt.savefig(f'{OUT_DIR}/6_cap_lech_nhat.png', dpi=120); plt.show()

## 7. Năm bộ ảnh minh hoạ
Mỗi hàng gồm: ảnh thường | ảnh hồng ngoại | nhãn (bảng màu gốc của FMB) | ảnh thường chồng cạnh IR (đỏ). Ảnh chồng cạnh giúp thấy ngay IR và VIS có khớp nhau không. Các mẫu được chọn trải đều từ tối nhất đến sáng nhất để có cả ảnh ngày và đêm. Chú thích tên lớp ở cuối hình.

In [ ]:
order = df_img.sort_values('vis_mean').reset_index(drop=True)
picks = order.iloc[np.linspace(0, len(order) - 1, N_SAMPLES).astype(int)]
print('Bảng màu:', 'gốc của FMB (từ thư mục color)' if OFFICIAL_PALETTE is not None else 'ngẫu nhiên (mục 4b không tìm được bảng màu gốc)')

fig, axes = plt.subplots(N_SAMPLES, 4, figsize=(16, 3.1 * N_SAMPLES + 0.9), squeeze=False)
seen = set()
for r, (_, row) in enumerate(picks.iterrows()):
    p = path_of[(row.split, row['name'])]
    vis, ir = cv2.imread(p['vis']), cv2.imread(p['ir']); lab = read_label(p['label'])
    seen |= set(np.unique(lab).tolist())
    panels = [(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB), f"VIS  {row.split}/{row['name']}  (sáng TB {row.vis_mean:.0f})"),
              (cv2.cvtColor(ir, cv2.COLOR_BGR2RGB), 'IR'), (PALETTE[lab], 'Label'), (overlay_edges(vis, ir), 'VIS + cạnh IR (đỏ)')]
    for c, (im, t) in enumerate(panels):
        axes[r, c].imshow(im); axes[r, c].set_title(t, fontsize=9); axes[r, c].axis('off')
handles = [Patch(facecolor=PALETTE[v] / 255, edgecolor='k', lw=.3, label=cname(v)) for v in sorted(seen)]
fig.legend(handles=handles, loc='lower center', ncol=min(8, len(handles)), fontsize=9, frameon=False)
plt.tight_layout(rect=(0, 0.05, 1, 1)); plt.savefig(f'{OUT_DIR}/7_bo_anh_minh_hoa.png', dpi=130); plt.show()
display(picks[['split', 'name', 'vis_mean']].round(1))

## 8. Tóm tắt
Gom các số chính vào `8_tom_tat.json` để dán vào báo cáo hoặc slide.

In [ ]:
hc = df_al[df_al.tin_cay_cao]
summary = {
    'root': ROOT,
    'so_cap_theo_split': {r['split']: int(r['Cặp đủ']) for _, r in df_count.iterrows()},
    'so_loi_ghep_cap': int(len(df_pi)),
    'anh_hong': len(broken),
    'cap_lech_kich_thuoc': int((~same).sum()),
    'kich_thuoc_pho_bien_vis_wxh': [int(x) for x in df_img.groupby(['vis_w', 'vis_h']).size().idxmax()],
    'ir_3_kenh_giong_nhau': f'{n_ir3}/{len(df_img)}',
    'gia_tri_label': values,
    'color_la_nhan_to_mau': OFFICIAL_PALETTE is not None,
    'lop_thieu_giua_cac_split': missing,
    'lop_hiem_duoi_0_5pct_pixel': df_cls.loc[rare, 'lớp'].tolist(),
    'ro_du_lieu': {'trung_tuyet_doi_train_test': n_exact,
                   'anh_test_ro_chot_bang_mat': leak_test,
                   'ty_le_test_ro_pct': round(100 * len(leak_test) / len(Nt), 2),
                   'so_anh_test_sau_khi_bo': int(len(Nt) - len(leak_test))},
    'can_chinh_IR_VIS': {'so_cap': int(len(df_al)),
                         'trung_vi_dx_tin_cay_cao': round(float(hc.dx.median()), 2),
                         'trung_vi_dy_tin_cay_cao': round(float(hc.dy.median()), 2),
                         'trung_vi_lech_px_tin_cay_cao': round(float(hc.lech_px.median()), 2),
                         'P90_lech_px_tin_cay_cao': round(float(hc.lech_px.quantile(0.9)), 2),
                         'pct_tren_10px_tin_cay_cao': round(float(100 * (hc.lech_px > 10).mean()), 2),
                         'trung_vi_lech_px_tat_ca': round(float(df_al.lech_px.median()), 2),
                         'tuong_quan_tin_cay_do_sang': float(corr_tc_sang)},
    'anh_minh_hoa': [f"{r.split}/{r['name']}" for _, r in picks.iterrows()],
}
json.dump(summary, open(f'{OUT_DIR}/8_tom_tat.json', 'w'), ensure_ascii=False, indent=2)
print(json.dumps(summary, ensure_ascii=False, indent=2))
print('\nFile đã lưu:', sorted(os.listdir(OUT_DIR)))

## Kết luận audit (chốt 3/11, dùng cho báo cáo)
Đối chiếu với số in ra ở trên sau khi chạy lại; nếu khác thì sửa ở đây.

1. **Số cặp và ghép cặp:** train 1.220 + test 280 = 1.500, khớp bài công bố. 0 lỗi ghép cặp, 0 ảnh hỏng, 0 cặp lệch kích thước, tất cả 800×600.
2. **Định dạng:** VIS 3 kênh uint8. IR lưu 3 kênh nhưng 3 kênh giống hệt → đọc 1 kênh. Label 1 kênh uint8, giá trị 0–14, không có 255.
3. **Thư mục `color`:** nhãn tô màu theo bảng màu gốc, mỗi id ứng đúng 1 màu; id 0 = đen = unlabeled.
4. **Lớp hiếm:** bicycle chỉ có ở 12 ảnh train và **0 ảnh test** → IoU bicycle trên test không xác định; theo quy ước SegMiF (NaN → 0, tính cả unlabeled) mIoU bị kéo xuống ~6,7% tương đối. Motorcycle và traffic light rất ít pixel.
5. **Rò dữ liệu:** 0 ảnh trùng hệt, nhưng **7/280 ảnh test (2,5%)** là khung liền kề của cùng đoạn video với ảnh train (2 cụm). Báo mIoU trên đủ 280 ảnh và trên 273 ảnh còn lại.
6. **Căn chỉnh IR–VIS:** nhóm tin cậy cao lệch trung vị ~1 px theo mỗi chiều, |lệch| trung vị ~3 px; các độ lệch lớn đều ở ảnh tối/sương (đo hỏng). Không có lệch hệ thống đáng kể.
7. **Minh hoạ:** test 00089 (VIS bị khói che, IR vẫn thấy 2 người) là ví dụ tốt cho phần động lực của đề tài.